# PhysLens on Colab A100 — checkpoint-safe Week A runner

**Purpose:** Run the Phase 5 Week A experiments (random-direction control, pre-registered SCAS sweep, PhysLens-OC 4-way contest) on a Colab Pro A100 with Google Drive mounted for checkpoint persistence.

**Runtime:** A100 (40GB) — `Runtime → Change runtime type → A100 GPU`.

**Survives disconnect:** every intermediate result is atomically written to `/content/drive/MyDrive/PhysLens/`. Re-running any cell picks up where it left off.

**Expected total wall-clock:** ~75 minutes of A100 time, ~15 compute units.

## Execution order

1. Mount Drive
2. Clone/pull repo
3. Install deps
4. Sync feature cache + PhysBench data from Drive (one-time upload)
5. **Gate 1:** Random-direction control (20 seeds)
6. **Gate 2:** SCAS α=3 sweep (pre-registered)
7. **Gate 3:** PhysLens-OC 4-way contest
8. **Gate 4:** Aggregate + kill-gate verdict

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/PhysLens')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
for sub in ['cache', 'results', 'logs', 'data']:
    (DRIVE_ROOT / sub).mkdir(exist_ok=True)

print('Drive mounted at', DRIVE_ROOT)
print('Contents:', sorted(p.name for p in DRIVE_ROOT.iterdir()))

# Verify GPU.
!nvidia-smi | head -10

## 2. Clone or pull repo

The repo lives on `/content` (ephemeral) for speed. Only checkpoints + cache + logs go to Drive.

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/Sonica-B/VLAs.git'
BRANCH = 'physics-steering'
REPO_DIR = '/content/VLAs'

if os.path.exists(REPO_DIR):
    print('Repo exists; pulling latest on', BRANCH)
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

%cd /content/VLAs
!git log --oneline -5

## 3. Install dependencies

Uses Colab's preinstalled torch (CUDA 12.x). We add transformers-main, bitsandbytes, peft, qwen-vl-utils, and a few scientific deps.

In [ ]:
!pip install -q --upgrade pip
!pip install -q 'transformers>=4.48' 'accelerate>=0.33' 'bitsandbytes>=0.44' \
    'peft>=0.13' 'qwen-vl-utils' 'safetensors' 'sentencepiece' 'protobuf' \
    'scikit-learn' 'scipy' 'h5py' 'Pillow' 'einops' 'timm'
# -e for editable install so local src/ changes apply without reinstall.
!pip install -q -e /content/VLAs

import torch, transformers, bitsandbytes as bnb
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'avail', torch.cuda.is_available())
print('transformers', transformers.__version__)
print('bitsandbytes', bnb.__version__)

## 4. Sync cache + data from Drive

**One-time step:** upload your local `cache/week1/features/` + `data/physbench/` to `PhysLens/cache/` and `PhysLens/data/` on Drive. Then this cell symlinks them into the repo.

**Required Drive layout (upload this manually the first time):**

```
PhysLens/
├── cache/week1/features/          # Week 1 cached activations per model
│   └── qwen3-vl-8b/
│       └── train/
│           ├── index.json
│           └── ...h5 files
├── data/physbench/                 # PhysBench val + test + answers
│   ├── val.json
│   ├── test.json
│   ├── test_answer.json
│   └── image/, video/
└── results/week4/                  # written by this notebook
```

**Tip:** Upload once via a zipfile, then `unzip -n` here — faster than 1000s of small Drive syncs.

In [ ]:
import pathlib, shutil, os

DRIVE_CACHE = pathlib.Path('/content/drive/MyDrive/PhysLens/cache')
DRIVE_DATA = pathlib.Path('/content/drive/MyDrive/PhysLens/data')
DRIVE_RESULTS = pathlib.Path('/content/drive/MyDrive/PhysLens/results')
DRIVE_LOGS = pathlib.Path('/content/drive/MyDrive/PhysLens/logs')

# Sanity check the Drive layout.
required_cache = DRIVE_CACHE / 'week1' / 'features' / 'qwen3-vl-8b' / 'train'
required_data = DRIVE_DATA / 'physbench' / 'val.json'

missing = []
if not required_cache.exists():
    missing.append(str(required_cache))
if not required_data.exists():
    missing.append(str(required_data))

if missing:
    print('MISSING ON DRIVE:')
    for m in missing:
        print(' ', m)
    print('\nUpload these from your local machine first, then rerun this cell.')
    print('Recommended: zip cache/week1 + data/physbench locally, upload the zip to PhysLens/, unzip here.')
else:
    # Symlink into the repo.
    def link(src: pathlib.Path, dst: pathlib.Path):
        if dst.is_symlink() or dst.exists():
            if dst.is_symlink():
                dst.unlink()
            else:
                print('WARN:', dst, 'exists and is not a symlink; leaving as-is')
                return
        dst.parent.mkdir(parents=True, exist_ok=True)
        os.symlink(src, dst)
        print('symlink', dst, '->', src)

    link(DRIVE_CACHE, pathlib.Path('/content/VLAs/cache'))
    link(DRIVE_DATA, pathlib.Path('/content/VLAs/data'))
    link(DRIVE_RESULTS, pathlib.Path('/content/VLAs/results_drive'))
    link(DRIVE_LOGS, pathlib.Path('/content/VLAs/logs_drive'))
    print('\nDrive sync OK. Ready to run.')

In [ ]:
# Configure output dirs: write directly to Drive-backed paths so disconnects
# do not lose partial progress.
import os, pathlib

os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/content/hf_cache'
pathlib.Path('/content/hf_cache').mkdir(exist_ok=True)

# If the user has HF_TOKEN in Colab Secrets, use it (needed for Qwen2.5/LLaVA).
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = token
        print('HF_TOKEN loaded from Colab Secrets')
except Exception as e:
    print('No HF token in Secrets (OK for Qwen3-VL which is public):', e)

## 5. Smoke test (3 seeds, 20 samples)

Run this FIRST before burning compute on the full 20-seed pass. If this works end-to-end, the full run is trustworthy.

In [ ]:
%cd /content/VLAs
!python scripts/week3_scas_random_control.py \
    --model qwen3-vl-8b \
    --seeds 3 --max-samples 20 --alpha 5.0 \
    --cache-dir /content/VLAs/cache/week1 \
    --data-dir /content/VLAs/data/physbench \
    --output-dir /content/VLAs/results_drive/week4/random_control_smoke \
    --log-dir /content/VLAs/logs_drive \
    --include-baseline

## 6. Gate 1 — Random-direction control (full 20 seeds, val n=200)

**Estimated:** ~70 min. **Compute:** ~14 units.

**Checkpoint behavior:** each seed is a standalone atomic write. Ctrl-C or disconnect is safe; re-run this cell to resume from the next uncomputed seed.

In [ ]:
%cd /content/VLAs
!python scripts/week3_scas_random_control.py \
    --model qwen3-vl-8b \
    --seeds 20 --alpha 5.0 \
    --cache-dir /content/VLAs/cache/week1 \
    --data-dir /content/VLAs/data/physbench \
    --output-dir /content/VLAs/results_drive/week4/random_control \
    --log-dir /content/VLAs/logs_drive \
    --include-baseline

In [ ]:
# Inspect the summary — this is the Gate 1 verdict.
import json, pathlib
summary_path = pathlib.Path('/content/VLAs/results_drive/week4/random_control/qwen3-vl-8b/random_control_summary.json')
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    for k in ['n_seeds', 'baseline_acc_quant', 'random_acc_quant_median',
              'random_delta_quant_median', 'random_delta_quant_ci95',
              'observed_scas_delta_quant', 'kill_threshold_delta_quant',
              'kill_gate_fired', 'verdict']:
        print(f'{k:34s} = {summary.get(k)}')
else:
    print('No summary yet — finish the full run first.')

## 7. Gate 2 — SCAS α=3 sweep (pre-registered, previously never run)

α=3 was specified in Phase 4 but the actual sweep only tested 0,5,10,15,20,30,50. This closes that pre-registration gap.

In [ ]:
%cd /content/VLAs
!python scripts/week3_scas_sweep.py \
    --model qwen3-vl-8b \
    --alphas 0 3 5 \
    --method amplify \
    --pca-split train \
    --eval-split val \
    --cache-dir /content/VLAs/cache/week1 \
    --data-dir /content/VLAs/data/physbench \
    --output-dir /content/VLAs/results_drive/week4/scas_prereg \
    --log-dir /content/VLAs/logs_drive

## 8. Gate 3 — PhysLens-OC 4-way contest

Compares:
  1. `amplify` (current SCAS headline)
  2. `contrast` (PhysLens-OC — novel variant)
  3. random-direction (already run in Gate 1; reused)
  4. `qual-contrast` (positive control: should hurt quant)

We run (1), (2), (4) here. (3) is the random baseline already on disk.

In [ ]:
# Amplify (redundant with Gate 2 but saved to this dir for apples-to-apples).
%cd /content/VLAs
!python scripts/week3_scas_sweep.py \
    --model qwen3-vl-8b --alphas 0 5 --method amplify \
    --pca-split train --eval-split val \
    --cache-dir /content/VLAs/cache/week1 \
    --data-dir /content/VLAs/data/physbench \
    --output-dir /content/VLAs/results_drive/week4/oc_contest/amplify \
    --log-dir /content/VLAs/logs_drive

In [ ]:
# Contrast (PhysLens-OC). Uses quant_centroid - qual_centroid projected onto V_low.
%cd /content/VLAs
!python scripts/week3_scas_sweep.py \
    --model qwen3-vl-8b --alphas 0 5 --method contrast \
    --pca-split train --eval-split val \
    --cache-dir /content/VLAs/cache/week1 \
    --data-dir /content/VLAs/data/physbench \
    --output-dir /content/VLAs/results_drive/week4/oc_contest/contrast \
    --log-dir /content/VLAs/logs_drive

In [ ]:
# Qual-contrast positive control: flip the centroid subtraction.
# This should HURT quant if contrast genuinely targets physics.
# Implemented via a tiny monkeypatch that subtracts the SAME contrast vector
# (equivalent to alpha<0) — just run contrast with alpha=-5.
%cd /content/VLAs
!python scripts/week3_scas_sweep.py \
    --model qwen3-vl-8b --alphas 0 -5 --method contrast \
    --pca-split train --eval-split val \
    --cache-dir /content/VLAs/cache/week1 \
    --data-dir /content/VLAs/data/physbench \
    --output-dir /content/VLAs/results_drive/week4/oc_contest/qual_contrast \
    --log-dir /content/VLAs/logs_drive

## 9. Final aggregation

Build the headline table and write the Gate 1-4 verdicts to Drive. Report numbers back to Notion manually.

In [ ]:
import json, pathlib

def load(p):
    p = pathlib.Path(p)
    return json.loads(p.read_text()) if p.exists() else None

base = '/content/VLAs/results_drive/week4'

# Random-direction summary.
rc = load(f'{base}/random_control/qwen3-vl-8b/random_control_summary.json')
# Pre-registered alpha sweep.
prereg = load(f'{base}/scas_prereg/scas_sweep_qwen3-vl-8b.json')
# OC contest.
amp = load(f'{base}/oc_contest/amplify/scas_sweep_qwen3-vl-8b.json')
con = load(f'{base}/oc_contest/contrast/scas_sweep_qwen3-vl-8b.json')
qual = load(f'{base}/oc_contest/qual_contrast/scas_sweep_qwen3-vl-8b.json')

def get_sweep_delta(sweep_json, target_alpha):
    """Return (delta_quant, delta_qual) for target_alpha vs alpha=0 baseline."""
    if sweep_json is None:
        return (None, None)
    baseline = next((s for s in sweep_json['sweep'] if s['alpha'] == 0.0), None)
    target = next((s for s in sweep_json['sweep'] if abs(s['alpha'] - target_alpha) < 1e-6), None)
    if baseline is None or target is None:
        return (None, None)
    return (target['acc_quant'] - baseline['acc_quant'],
            target['acc_qual'] - baseline['acc_qual'])

print('='*78)
print(' PHYSLENS WEEK A — HEADLINE TABLE (Qwen3-VL-8B, PhysBench val n=200)')
print('='*78)
print(f'  {"condition":<32} {"alpha":>8} {"d_quant":>10} {"d_qual":>10}')
print(f'  {"-"*62}')
if prereg is not None:
    for a in [3.0, 5.0]:
        dq, dl = get_sweep_delta(prereg, a)
        if dq is not None:
            print(f'  {"SCAS amplify (pre-reg)":<32} {a:>8.1f} {dq:+.4f} {dl:+.4f}')
if amp is not None:
    dq, dl = get_sweep_delta(amp, 5.0)
    if dq is not None: print(f'  {"SCAS amplify (OC contest)":<32} {5.0:>8.1f} {dq:+.4f} {dl:+.4f}')
if con is not None:
    dq, dl = get_sweep_delta(con, 5.0)
    if dq is not None: print(f'  {"PhysLens-OC contrast":<32} {5.0:>8.1f} {dq:+.4f} {dl:+.4f}')
if qual is not None:
    dq, dl = get_sweep_delta(qual, -5.0)
    if dq is not None: print(f'  {"qual-contrast (pos. control)":<32} {-5.0:>8.1f} {dq:+.4f} {dl:+.4f}')
if rc is not None:
    print(f'  {"random-direction (median)":<32} {5.0:>8.1f} {rc["random_delta_quant_median"]:+.4f} {rc["random_delta_qual_median"]:+.4f}')
    ci = rc['random_delta_quant_ci95']
    print(f'    random 95% CI on d_quant: [{ci[0]:+.4f}, {ci[1]:+.4f}]  (n_seeds={rc["n_seeds"]})')
print('='*78)

# Verdict.
print()
print('GATES:')
if rc is not None:
    print(f'  Gate 1 (random control): kill_fired={rc["kill_gate_fired"]}')
    print(f'    {rc["verdict"]}')
else:
    print('  Gate 1 (random control): not yet run')

verdict_path = pathlib.Path(f'{base}/week_a_verdict.json')
verdict = {
    'gate_1_random_control': rc,
    'prereg_sweep': prereg,
    'oc_amplify': amp,
    'oc_contrast': con,
    'oc_qual_contrast': qual,
}
import os as _os
tmp = verdict_path.with_suffix('.tmp')
with open(tmp, 'w') as f:
    json.dump(verdict, f, indent=2, default=str)
    f.flush(); _os.fsync(f.fileno())
_os.replace(tmp, verdict_path)
print(f'\nWrote verdict: {verdict_path}')

## 10. Keep-alive helper (optional)

Colab disconnects after 90 min idle. Running this in a background cell clicks the notebook periodically. **Do NOT abuse** — Colab can ban accounts that run keep-alives while not using compute. Only enable during active A100 runs.

In [ ]:
# Uncomment to enable keep-alive JS in the browser (clicks 'Connect' periodically).
# from IPython.display import Javascript, display
# display(Javascript('''
# function ClickConnect(){
#   console.log('keep-alive ping');
#   document.querySelector('colab-toolbar-button#connect').click()
# }
# setInterval(ClickConnect, 60000)
# '''))
print('(keep-alive JS commented out — uncomment only during active compute)')